# Unix-terminal. Настройка окружения и установка пакетов


## Мотивация

Python-проект — это не только код, но и версия интерпретатора, точные версии пакетов и их зависимостей. Пакет с тем же именем после обновления может вести себя иначе: например, код вокруг `timm` ожидает четыре промежуточные карты признаков модели, а получает три и ломается уже во время обучения. Отдельное окружение и зафиксированные зависимости позволяют воспроизвести рабочий запуск на другой машине и обновлять проект осознанно.

Вторая половина темы — про то, где система вообще берёт то, что вы запускаете: почему `command not found` и `ModuleNotFoundError` — это разные поломки с разным лечением, и почему программа, отлично работающая из вашего терминала, не стартует как служба.


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

История про `timm` — не выдуманная: так ломается обучение через час после старта, и хуже всего то, что код при этом **импортируется без ошибок**. Обновилась библиотека, форма выхода стала другой, а падение прилетает в середине эпохи.

Отсюда и правило: окружение проекта — часть эксперимента. «У меня работает» — не результат, пока рядом нет `pyproject.toml`/`uv.lock` (или хотя бы `requirements.txt`), по которым окружение восстанавливается на чужой машине.

</details>


> **Как устроен семинар.** Все команды выполняются **на вашей виртуальной машине, в домашнем
> каталоге** — в `~/seminar-05/`. Каталог `/tmp` не используем: он вычищается при перезагрузке, а
> окружения и проекты этого занятия нужны вам и после него.
>
> Каждая ячейка `%%bash` запускает **новый** shell, поэтому переменные и `cd` из соседней ячейки в
> ней не действуют — в начале ячеек мы заново переходим в рабочий каталог. Это же свойство спасает:
> сломанный в ячейке `PATH` никуда не утечёт.
>
> Свёрнутые блоки **🎙 Заметка преподавателя** — то, что рассказывается вслух на занятии;
> разворачивайте их потом, при подготовке к защите.


## 1. Переменные оболочки, `export` и `source`

`NAME=value` создаёт переменную в текущей оболочке. `export NAME=value` добавляет её в окружение: значение получают программы и дочерние оболочки, запущенные после `export`. Запись `NAME=value command` передаёт значение только процессу `command`.

`source FILE` выполняет команды из файла в текущей оболочке, поэтому смена каталога и значения переменных сохраняются. `bash FILE` запускает отдельную дочернюю оболочку: её переменные исчезнут после завершения, а родительская оболочка не изменится.

`env` и `printenv` показывают окружение, которое получат дочерние процессы. Переменные без `export` в этот список не входят.

Заведём рабочий каталог занятия и файл с настройками учебного проекта: одна переменная в нём объявлена без `export`, другая — с `export`.


In [ ]:
%%bash
rm -rf ~/seminar-05/course-app          # чистый старт: ячейку можно перезапускать
mkdir -p ~/seminar-05/course-app
cd ~/seminar-05/course-app || exit 1    # || exit 1 — не выполнять остальное, если каталога нет

# 'EOF' в кавычках: heredoc пишется в файл как есть, $HOME не подставится сейчас
cat > course.env <<'EOF'
COURSE_NAME='course-app'
export COURSE_DATA="$HOME/seminar-05/course-app/data"
EOF


Файл написан, но пока это просто текст на диске — никаких переменных он ещё не создал. Подключим его
к текущей оболочке через `source` и посмотрим, что увидит она сама, а что — запущенный из неё
дочерний процесс.


In [ ]:
%%bash
cd ~/seminar-05/course-app || exit 1
source ./course.env                                 # выполняем файл в ТЕКУЩЕЙ оболочке

echo "current: $COURSE_NAME"                        # видна: source не порождает новый процесс
bash -c 'echo "child:   ${COURSE_NAME:-missing}"'   # missing — без export значение не передаётся
bash -c 'echo "child:   $COURSE_DATA"'              # а это значение дошло: оно экспортировано
env | grep '^COURSE_' || true                       # в окружении только экспортированная переменная


Разница видна в третьей и четвёртой строках вывода: обе переменные существуют в текущей оболочке,
но дочерний процесс получил только экспортированную. `env` показывает ровно то, что достанется
детям, — поэтому полный вывод `env` нельзя выкладывать в чат или в issue: в нём бывают токены.


#### ❓ **Вопрос**: В `course.env` переменная `COURSE_NAME` задана без `export`, а `COURSE_DATA` — с `export`. Что увидят текущая оболочка и запущенный из неё `bash -c` после `source course.env`?

<details>

<summary><strong>Ответ</strong></summary>

Текущая оболочка увидит обе переменные, потому что `source` выполняет файл в ней. Дочерний `bash -c` получит только `COURSE_DATA`: без `export` значение `COURSE_NAME` не передаётся дочернему процессу — это и показали строки `child:` в выводе.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

`source FILE` и `. FILE` — одно и то же; точка короче и работает в любом POSIX-shell, `source` —
удобное имя из bash. Классическая ошибка новичка: написать скрипт `setenv.sh` с `export`, запустить
его как `./setenv.sh` — и удивиться, что переменных нет. Скрипт отработал в дочернем процессе и
унёс окружение с собой; нужен `source ./setenv.sh`.

Ни обычная переменная, ни `export` не переживают закрытие терминала: новый терминал — новый процесс
shell. Постоянные настройки пишут в стартовый файл bash. Интерактивный bash, запущенный не как
login shell, читает `~/.bashrc`; login shell — первый доступный файл из `~/.bash_profile`,
`~/.bash_login`, `~/.profile`. Какой из них сработает, зависит от способа запуска оболочки, поэтому
незнакомый стартовый файл сначала читают, а потом дописывают.

</details>


## 2. `PATH` — где оболочка ищет программу

`PATH` — список каталогов с исполняемыми файлами, разделённых двоеточиями. Когда вы набрали имя без `/`, оболочка проверяет эти каталоги слева направо и запускает первый найденный файл. `which name` (или встроенный `command -v name`) показывает, какой именно файл будет выбран.

Полный путь поиска не требует: `/usr/bin/ls` запустится даже при сломанном `PATH`. Это полезный временный обход, но причину всё равно исправляют.


Как оболочка ищет программу по PATH


Проверим это на своей команде. Сначала — обстановка: каталог `bin` со скриптом, который печатает
значение переменной окружения.


In [ ]:
%%bash
mkdir -p ~/seminar-05/course-app/bin
cat > ~/seminar-05/course-app/bin/course-info <<'EOF'
#!/usr/bin/env bash
echo "course=${COURSE_NAME:-unset}"
EOF
chmod +x ~/seminar-05/course-app/bin/course-info   # без бита x оболочка файл не запустит
ls -l ~/seminar-05/course-app/bin/course-info      # видно rwx — файл исполняемый


Файл есть, но по имени `course-info` пока не запускается: его каталога нет в `PATH`. Добавим этот
каталог **в начало** списка — и команда появится.


In [ ]:
%%bash
export PATH="$HOME/seminar-05/course-app/bin:$PATH"   # свой каталог первым, старое значение сохранили
which course-info                                     # какой файл выберет оболочка
export COURSE_NAME='course-app'                       # скрипт читает переменную из окружения
course-info                                           # запуск по имени, без пути


Обратите внимание на `:$PATH` в конце: мы **дописали** свой каталог к прежнему списку, а не заменили
его. Посмотрим, что бывает, если старое значение потерять. Ячейка `%%bash` — отдельный процесс,
поэтому ломаем `PATH` безопасно: на вашу оболочку это не повлияет.


In [ ]:
%%bash
PATH="$HOME/bin"                       # затираем список целиком — так делать не надо
command -v ls || echo 'ls не найден: в PATH не осталось системных каталогов'
/usr/bin/ls ~/seminar-05/course-app    # полный путь работает и при сломанном PATH


#### ❓ **Вопрос**: После `PATH="$HOME/bin"` команда `ls` перестала находиться. Как вернуть поиск системных команд и как запустить `ls` прямо сейчас, пока `PATH` ещё не исправлен?

<details>

<summary><strong>Ответ</strong></summary>

Прямо сейчас — по полному пути: `/usr/bin/ls`, как в последней строке ячейки: поиск по `PATH` для этого не нужен. Вернуть поиск можно, восстановив прежнее значение (или просто открыв новый терминал — стартовый файл задаст `PATH` заново). А добавлять свой каталог следует так, как в предыдущей ячейке: `export PATH="$HOME/bin:$PATH"`, не отбрасывая старое значение.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Две вещи, на которые студенты натыкаются позже.

Первая: bash запоминает найденные пути в хэше, поэтому после установки новой версии программы в
каталог, стоящий раньше в `PATH`, старый терминал может продолжать запускать прежнюю. Лечится
`hash -r` (или новым терминалом), и это частая причина «я же обновил, а версия старая».

Вторая: текущий каталог `.` в `PATH` — дыра в безопасности. Достаточно распаковать чужой архив, где
лежит файл `ls`, и в этом каталоге вы запустите его вместо системного. Поэтому в Unix `.` в `PATH`
не добавляют, а свои скрипты кладут в `~/bin` или `~/.local/bin`.

</details>


## 3. Интерпретатор, `-m`, `venv` и `virtualenv`


### Что делает `python -m`

Обычный запуск `python script.py` выполняет файл по указанному пути. Запуск `python -m module` просит **выбранный Python** найти модуль так же, как при импорте, и выполнить его как программу. Если указан обычный модуль, выполняется его `.py`-файл; если пакет — файл `__main__.py` внутри пакета. Отдельный модуль пакета запускают как `python -m package.module`.

Поэтому `python -m pip` запускает модуль `pip`, установленный именно у выбранного `python`, а не случайную одноимённую команду из другого места в `PATH`.

Где Python ищет модули для импорта, задаёт переменная `PYTHONPATH` (плюс каталоги самого интерпретатора и окружения). Заведём маленький пакет и посмотрим на это руками.


In [ ]:
%%bash
rm -rf ~/seminar-05/python-demo         # чистый старт: ячейку можно перезапускать
mkdir -p ~/seminar-05/python-demo/course_app
cd ~/seminar-05/python-demo/course_app || exit 1

cat > say_hello_module.py <<'PY'
print("Hello world")
PY


Модуль лежит в `~/seminar-05/python-demo/course_app/`. Запустим его **из другого каталога** — сначала
как есть, потом с указанным `PYTHONPATH`.


In [ ]:
%%bash
cd ~ || exit 1                                          # рядом никакого course_app нет
python3 -m course_app.say_hello_module 2>&1 | tail -1   # ModuleNotFoundError: искать негде

# PYTHONPATH=... перед командой — значение только для этого запуска
PYTHONPATH=~/seminar-05/python-demo python3 -m course_app.say_hello_module


Первая строка — типичный `ModuleNotFoundError`, вторая — тот же код, найденный по `PYTHONPATH`.

Теперь добавим в пакет файл `__main__.py`: он выполняется, когда `-m` указывает на пакет целиком.


In [ ]:
%%bash
cd ~/seminar-05/python-demo/course_app || exit 1
cat > __main__.py <<'PY'
from . import say_hello_module
PY
ls ~/seminar-05/python-demo/course_app   # два наших файла (__pycache__ — кэш прошлого запуска)


In [ ]:
%%bash
cd ~ || exit 1
PYTHONPATH=~/seminar-05/python-demo python3 -m course_app   # пакет целиком → выполняется __main__.py


Тот же механизм работает и для установленных пакетов — например, `pip` — с одной оговоркой: модуль
берётся у **того** интерпретатора, которого вы позвали.


In [ ]:
%%bash
python3 -m json.tool <<< '{"b":1,"a":2}'   # модуль стандартной библиотеки как программа
python3 -m pip --version 2>&1 | tail -1    # pip ИМЕННО этого python (или сообщение, что его нет)


Если вторая строка сказала `No module named pip` — это нормально: у системного Python в Ubuntu `pip`
ставится отдельным пакетом `python3-pip`. Важно другое: `python3 -m pip` спрашивает pip у
конкретного интерпретатора и потому никогда не поставит пакет «не туда», в отличие от команды `pip`,
выбранной по `PATH`.


#### ❓ **Вопрос**: Почему `python3 -m pip install X` надёжнее, чем просто `pip install X`?

<details>

<summary><strong>Ответ</strong></summary>

`pip` — обычная команда, её выбирает оболочка по `PATH`, и это может оказаться pip совсем другого интерпретатора (например, системного, когда вы работаете в окружении). `python3 -m pip` сначала выбирает интерпретатор, а модуль `pip` берётся уже у него — пакет гарантированно ставится туда, куда вы смотрите. Ровно так же в демке `-m` нашёл `course_app` только у того `python3`, которому мы задали `PYTHONPATH`.

</details>


### Зачем несколько Python и несколько `.venv`

На одной машине могут одновременно требоваться разные версии Python: старый проект ещё работает с Python 3.10, новый использует возможности 3.12, а версия на рабочем сервере должна совпадать с проверенной. Виртуальное окружение создаётся **на основе конкретного интерпретатора** и не превращает Python 3.11 в Python 3.12.

Даже проекты на одной версии Python получают разные `.venv`: одному нужен пакет `library==1`, другому — несовместимый `library==2`. Глобальная или пользовательская установка смешивает зависимости проектов: обновление одного пакета может сломать соседний код, а рабочее окружение будет трудно воспроизвести на другой машине.

```text
машина
├── system Python 3.11          ← нужен ОС и системным утилитам
├── Python 3.12 под управлением uv
│   ├── project-a/.venv         ← requests 2.31
│   └── project-b/.venv         ← requests 2.32
└── Python 3.13 под управлением uv
    └── experiment/.venv        ← отдельный набор пакетов
```

`uv python find 3.12` показывает путь к подходящему интерпретатору. Создадим окружение на выбранной версии — и убедимся, что пользоваться им можно и без активации.


In [ ]:
%%bash
rm -rf ~/seminar-05/demo-venv                    # чистый старт: ячейку можно перезапускать
uv venv --python 3.12 ~/seminar-05/demo-venv     # окружение на основе Python 3.12
~/seminar-05/demo-venv/bin/python -V             # обращаемся к нему напрямую, без активации


Активация ничего не устанавливает и ничему не «переключает версию»: она лишь ставит каталог `bin`
окружения первым в `PATH` — то самое правило первого совпадения из раздела 2.


Что делает source .venv/bin/activate: было и стало


In [ ]:
%%bash
which python3                                       # до активации — системный интерпретатор
source ~/seminar-05/demo-venv/bin/activate          # source, а не bash: меняем текущую оболочку
which python                                        # теперь python — из окружения
echo "$PATH" | cut -d: -f1                          # первый элемент PATH — bin окружения
deactivate                                          # возвращает PATH как было
which python3


#### ❓ **Вопрос**: Проект A требует Python 3.11 и `library==1`, проект B — Python 3.12 и `library==2`. Достаточно ли двух `.venv`, созданных системным Python 3.11? Почему?

<details>

<summary><strong>Ответ</strong></summary>

Нет. Разные `.venv` изолируют версии `library`, но оба окружения останутся на Python 3.11: в демке `uv venv --python 3.12` явно выбирал интерпретатор, а `activate` только правил `PATH` и версию не менял. Для проекта B сначала нужен интерпретатор 3.12, а затем отдельная `.venv`, созданная на его основе.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

`activate` — обычный shell-скрипт, его можно открыть и прочитать: там правка `PATH`, сохранение
старого значения и функция `deactivate`, которая всё возвращает. Никакой магии; именно поэтому
активация действует только на ту оболочку, где вы её выполнили, — и совершенно не действует на
службу systemd (к этому вернёмся в разделе 7).

Историческая справка: `virtualenv` появился раньше и был внешним пакетом, потом идею внесли в
стандартную библиотеку как модуль `venv` (Python 3.3). Сегодня `python -m venv .venv` есть в любом
Python, а `uv` делает то же самое, только быстрее и умеет ещё и сам скачать нужную версию Python.

Важная оговорка: `.venv` — это **не** контейнер. Он изолирует Python-пакеты и не изолирует
системные библиотеки: `libcudnn` или `ffmpeg` из окружения не появятся.

</details>


## 4. Три пути поиска: `PATH`, `PYTHONPATH`, `LD_LIBRARY_PATH`

Все три переменные выглядят одинаково — каталоги через двоеточие — и потому их постоянно путают. Разница в том, **кто и когда** их читает: `PATH` читает оболочка, когда вы набрали команду; `PYTHONPATH` — интерпретатор Python при `import`; `LD_LIBRARY_PATH` — динамический загрузчик `ld.so`, когда запущенная программа подгружает библиотеки `.so`.

Практическая польза от этого различия — по тексту ошибки сразу понятно, куда смотреть.


Три списка путей: PATH, PYTHONPATH, LD_LIBRARY_PATH


Посмотрим на третий слой, который на семинарах обычно остаётся невидимым. Почти любая программа
собрана не целиком: часть кода лежит в общих библиотеках, которые подгружаются при старте.


In [ ]:
%%bash
ldd /usr/bin/head | head -3   # какие .so нужны программе и где загрузчик их нашёл
ldconfig -p | head -3         # общесистемный кэш библиотек: имя => путь
ldconfig -p | wc -l           # сколько библиотек в кэше /etc/ld.so.cache знает система


`ldd` отвечает на вопрос «что нужно этой программе и откуда это взялось», `ldconfig -p` печатает
общесистемный кэш `/etc/ld.so.cache` — список библиотек, известных системе. Кэш строит команда
`ldconfig` (от root) после установки новых библиотек; обычно это делает за вас пакетный менеджер.

Переменная `LD_LIBRARY_PATH` вклинивается **перед** кэшем: её каталоги проверяются первыми. Убедимся,
что порядок именно такой.


In [ ]:
%%bash
export LD_LIBRARY_PATH=/nonexistent               # каталог первый в очереди, но библиотек в нём нет
head -1 /etc/hostname                             # программа работает: загрузчик пошёл дальше по кэшу

# LD_DEBUG=libs просит загрузчик рассказать, куда он заглядывает
LD_DEBUG=libs head -1 /etc/hostname 2>&1 | grep -m2 'trying file'


В последних строках видно, что загрузчик сначала честно пытается открыть файл в `/nonexistent`, и
только не найдя — берёт библиотеку из кэша. Отсюда и правило: `LD_LIBRARY_PATH` — временный обход на
один запуск (например, чтобы подсунуть свежую сборку библиотеки), а не строка в `~/.bashrc`.
Постоянное место для библиотек — пакет ОС или каталог, добавленный в конфигурацию `ldconfig`.


#### ❓ **Вопрос**: Программа падает с `error while loading shared libraries: libfoo.so.1: cannot open shared object file`. Какая из трёх переменных здесь ни при чём и чем проверять?

<details>

<summary><strong>Ответ</strong></summary>

Не при чём `PYTHONPATH` (речь не про импорт модулей) и `PATH`: сама программа как раз нашлась и запустилась — иначе была бы `command not found`. Ошибку выдал загрузчик `ld.so`, значит смотреть надо на библиотеки: `ldd путь/к/программе` покажет, какая зависимость помечена `not found`, `ldconfig -p | grep libfoo` — знает ли о ней система. Дальше либо ставим нужный пакет ОС, либо на один запуск указываем каталог через `LD_LIBRARY_PATH`.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

`LD_LIBRARY_PATH` — классический источник «работало вчера». Кто-то однажды прописал его в
`~/.bashrc`, чтобы запустить старую сборку CUDA, и через полгода половина программ в системе стала
подхватывать чужие библиотеки: у переменной наивысший приоритет, и действует она на **всё**, что
запущено из этой оболочки. Правильный ответ почти всегда — поставить библиотеку пакетом или собрать
программу с `RPATH`, где путь зашит в сам бинарник.

Про `ldd` есть важная деталь для будущего: на незнакомом бинарнике `ldd` его фактически запускает
через загрузчик, так что `ldd` на скачанном непонятно откуда файле — не самая безопасная идея.
Безопасный вариант — `objdump -p file | grep NEEDED`.

</details>


## 5. Зависимости Python


### Прямые, транзитивные, ограничения и lock-файл

Допустим, код проекта импортирует `requests`:

- **прямая зависимость** — `requests`, потому что её выбрал сам проект и записал в `pyproject.toml`;
- **транзитивные зависимости** — например, `urllib3`, `certifi`, `idna`: они нужны `requests`, хотя проект не выбирал их напрямую;
- **ограничение версии** — `requests>=2.31,<3`: диапазон версий, которые разрешает проект, а не уже установленная версия;
- **зафиксированное решение** — точные версии всех выбранных пакетов и источники их загрузки, записанные в `uv.lock`.

```text
course-app
└── requests >=2.31,<3           ← прямая зависимость
    ├── urllib3                  ← транзитивная
    ├── certifi                  ← транзитивная
    └── idna                     ← транзитивная

pyproject.toml ── uv выбирает версии ──> uv.lock ── uv sync ──> .venv
  намерение                         точный план             файлы среды
```

Одинаковое имя пакета не гарантирует одинаковое поведение разных версий. Код может успешно импортировать обновлённый пакет, но получить другой результат или форму данных уже во время работы. Поэтому в Git сохраняют `pyproject.toml` и `uv.lock`, а `.venv` пересоздают.


### `uv` — основной инструмент курса

`uv` управляет Python, окружением и зависимостями проекта:

- `uv init` создаёт основу проекта;
- `uv python pin 3.12` записывает требуемый Python в `.python-version`;
- `uv add PACKAGE` добавляет прямую зависимость в `pyproject.toml`, устанавливает её и обновляет `uv.lock`;
- `uv remove PACKAGE` удаляет прямую зависимость из `pyproject.toml` и обновляет `uv.lock`;
- `uv tree` показывает дерево прямых и транзитивных зависимостей;
- `uv lock --check` проверяет, соответствует ли lock-файл описанию проекта;
- `uv sync` приводит `.venv` в состояние, записанное в `pyproject.toml` и `uv.lock`;
- `uv lock --upgrade` обновляет все пакеты до последних допустимых версий;
- `uv run COMMAND` синхронизирует окружение и запускает команду внутри него.

`uv.lock` не обновляется только потому, что в реестре появилась новая версия. Обновление выполняют явно.

Пройдём весь путь на учебном проекте — от пустого каталога до запуска кода в собранном окружении.


In [ ]:
%%bash
rm -rf ~/seminar-05/project              # чистый старт: ячейку можно перезапускать
mkdir -p ~/seminar-05/project
cd ~/seminar-05/project || exit 1
uv init                                  # заготовка проекта: pyproject.toml, README, каталог с кодом
uv python pin 3.12                       # требование к версии Python → файл .python-version
ls -a                                    # что появилось в каталоге


`uv init` заодно заводит git-репозиторий и `.gitignore`, в котором `.venv` уже перечислен: окружение
в историю не кладут, его пересоздают по `pyproject.toml` и `uv.lock`.

Зависимостей у проекта пока нет — только описание. Добавим прямую зависимость с ограничением версии:
одна команда правит `pyproject.toml`, пересчитывает `uv.lock` и ставит пакеты в `.venv`.


In [ ]:
%%bash
cd ~/seminar-05/project || exit 1
uv add 'requests>=2.31,<3'   # ограничение → pyproject.toml, точная версия → uv.lock, файлы → .venv
cat pyproject.toml           # намерение проекта: диапазон допустимых версий


В `pyproject.toml` записан **диапазон**, а не версия. Точное решение — какая именно версия выбрана и
что она за собой потянула — лежит в `uv.lock`.


In [ ]:
%%bash
cd ~/seminar-05/project || exit 1
grep -n -A 3 'name = "requests"' uv.lock | head -8   # выбранная версия — фиксированная, не диапазон
uv tree                                              # прямые зависимости и всё, что пришло с ними


`uv tree` показывает и транзитивные зависимости — те, которые проект не выбирал, но получил вместе с
`requests`. Осталось запустить код в этом окружении: `uv run` сам сверит `.venv` с lock-файлом.


In [ ]:
%%bash
cd ~/seminar-05/project || exit 1
uv run python -c 'import sys; print(sys.executable)'                # интерпретатор — из .venv проекта
uv run python -c 'import requests; print(requests.__version__)'     # ровно та версия, что в uv.lock


#### ❓ **Вопрос**: В `pyproject.toml` разрешено `requests>=2.31,<3`, а в `uv.lock` уже выбрана конкретная версия. В реестре появилась более новая совместимая версия. Поставит ли её обычный `uv sync` на другой машине?

<details>

<summary><strong>Ответ</strong></summary>

Нет. `uv sync` использует сохранённый lock-файл и воспроизводит зафиксированное решение — ту самую версию, которую мы видели в `uv.lock` и которую напечатал `uv run`. Чтобы перейти на новые совместимые версии, lock-файл обновляют явно: `uv lock --upgrade` (или `--upgrade-package requests` для одного пакета).

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Почему lock-файл вообще нужен, если версии и так записаны в `pyproject.toml`: в `pyproject.toml`
записано **намерение** («любая 2.x, начиная с 2.31»), а воспроизводимость требует **факта** («ровно
2.32.3, вот её хэш»). Раньше эту роль пытался играть `pip freeze > requirements.txt`, но он
сваливает в один список и прямые, и транзитивные зависимости, не отличая одно от другого: через
полгода никто не помнит, что из этого проект действительно выбирал.

`uv` написан на Rust командой Astral (авторы `ruff`) и появился в 2024 году; в задачах курса он
интересен в первую очередь скоростью — окружение, которое `pip` собирал минуту, поднимается за
секунды, и пересоздать `.venv` с нуля перестаёт быть страшно.

</details>


## 6. Области установки и системные менеджеры

Выбор области зависит от того, кому нужна программа:

```text
Linux-система
├── system:  /usr/bin, /usr/lib       ← apt, dnf, pacman; для всех пользователей
├── user:    ~/.local/bin             ← личные CLI-инструменты, например uv tool
└── project: project/.venv            ← импортируемые зависимости конкретного проекта
```

- Библиотека, которую импортирует проект, добавляется в проект через `uv add`.
- Самостоятельную CLI-утилиту для одного пользователя можно установить через `uv tool install` или менеджер вроде Homebrew.
- Системная программа и её системные зависимости устанавливаются менеджером ОС: `apt`/`apt-get` в Debian и Ubuntu, `dnf` в Fedora/RHEL, `pacman` в Arch.

`apt update` обновляет локальный индекс доступных версий, а `apt upgrade` — установленные пакеты. `apt install` устанавливает пакет, `apt remove` удаляет его, `apt purge` дополнительно удаляет системные конфигурационные файлы пакета. `apt` удобен для интерактивной работы, а `apt-get` имеет более стабильный интерфейс для автоматизации. Всё, что меняет систему, требует `sudo`; чтение — не требует, с него и начнём.


In [ ]:
%%bash
mkdir -p ~/seminar-05/reports
# apt list только читает базу пакетов, sudo не нужен; 2> — предупреждение о нестабильном CLI
apt list --installed > ~/seminar-05/reports/installed.txt 2> ~/seminar-05/reports/apt-errors.txt
wc -l < ~/seminar-05/reports/installed.txt   # сколько пакетов уже стоит в системе
head -3 ~/seminar-05/reports/installed.txt   # имя/репозиторий, версия, архитектура, [installed]


Такой же список доступных обновлений даёт `apt list --upgradable`. Установка же (`sudo apt install
NAME`) на этом занятии не показывается: она меняет общую систему, и запускать её всем вместе на
рабочей машине не стоит — в задачах вы сделаете это на своей ВМ.

В Ubuntu рядом с `apt` живёт второй менеджер — **snap**. Snap-пакет приезжает вместе со своими
зависимостями и обновляется сам, поэтому им распространяют браузеры и IDE; каталог его команд —
`/snap/bin`. Полезные команды: `snap list` (что установлено), `snap info NAME` (сведения о пакете),
`sudo snap install NAME` (установка). Одна и та же программа может существовать и в apt, и в snap —
какая из них запустится, решает уже знакомое правило `PATH`, а покажет `which`.

И отдельно про Python: `pip install` **без активного окружения** в свежих Ubuntu и Debian просто
откажется работать — системный интерпретатор помечен как `externally-managed`. Это не поломка, а
защита: пакеты из PyPI не должны затирать файлы, которыми управляет `apt`.


#### ❓ **Вопрос**: Куда логичнее установить: `requests`, который импортирует один проект; `ruff`, используемый как личная CLI-утилита; системный `curl`? Когда `ruff` всё же стоит добавить в зависимости проекта?

<details>

<summary><strong>Ответ</strong></summary>

`requests` — в `.venv` проекта через `uv add` (его импортирует код). Личный `ruff` — как пользовательский инструмент, `uv tool install ruff`: он запускается как команда, а не импортируется. Системный `curl` — менеджером ОС, `sudo apt install curl`: им пользуются все пользователи и другие программы. Если же команда и версия `ruff` должны быть одинаковыми у всей команды и проверяться в CI, его добавляют в группу зависимостей проекта.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Почему в Ubuntu два менеджера сразу: apt раздаёт пакеты, собранные под конкретную версию
дистрибутива, поэтому свежий Firefox для старой Ubuntu собрать сложно, а snap несёт свои
зависимости с собой и обновляется у пользователя молча. За это его и ругают: дольше стартует,
занимает больше места, обновляется без спроса. Похожая идея у flatpak и AppImage.

Про `externally-managed`: правило описано в PEP 668 и появилось после многолетней традиции ломать
систему командой `sudo pip install`. Классика жанра — `sudo pip install --upgrade requests` на
Ubuntu: pip сносит системную версию, apt об этом не знает, и вместе с ней перестают работать
системные утилиты, написанные на Python.

</details>


## 7. Службы и журналы `systemd`

`systemd` запускает и контролирует фоновые службы системы. `systemctl` показывает их состояние и управляет запуском, а `journalctl` читает собранные журналы.

Основные команды:

- `status NAME` — текущее состояние и несколько последних сообщений;
- `start`, `stop`, `restart` — управление сейчас;
- `enable NAME` — включить автоматический запуск при старте системы;
- `cat NAME` — показать найденный unit-файл и дополнения;
- `list-units --type=service` — загруженные службы;
- `list-unit-files --type=service` — все установленные unit-файлы служб;
- `journalctl -u NAME -n 50` — последние 50 сообщений службы;
- `--no-pager` — вывести результат прямо в терминал.

`start` запускает службу сейчас, но не включает автозапуск. `enable` включает автозапуск, но сам по себе не запускает службу. `enable --now` делает оба действия.

Посмотрим на живую службу — `systemd-journald`, ту самую, которая собирает журналы. Все три команды ниже только читают состояние, менять его мы не будем.


In [ ]:
%%bash
service_name=systemd-journald
systemctl status "$service_name" --no-pager | head -4   # loaded/active — состояние прямо сейчас
systemctl cat "$service_name" | head -4                 # какой unit-файл нашёлся и где он лежит
journalctl -u "$service_name" -n 3 --no-pager           # последние сообщения самой службы


Если `journalctl` ответил `-- No entries --` или отказом — у вашего пользователя нет прав на чтение
чужих журналов: их читают члены групп `adm` и `systemd-journal` либо root через `sudo`.

Обратите внимание на вывод `systemctl cat`: кроме основного unit-файла там могут быть drop-in-файлы
(каталог `NAME.service.d/`) — отдельные кусочки конфигурации, которые дописывают или переопределяют
настройки основного.


#### ❓ **Вопрос**: Что показали `systemctl status`, `systemctl cat` и `journalctl -u` для одной и той же службы? И запустит ли `systemctl enable` остановленную службу немедленно?

<details>

<summary><strong>Ответ</strong></summary>

`status` — текущее состояние (`loaded`, `active`) и несколько последних сообщений; `cat` — найденный unit-файл, его путь и drop-in-дополнения; `journalctl -u` — журнал именно этой службы. `enable` немедленно ничего не запускает: он только настраивает автозапуск при старте системы. Запустить сейчас — `systemctl start`, сделать оба действия — `systemctl enable --now`.

</details>


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

До systemd службами управляли скрипты SysV init: каталог `/etc/init.d`, номера для порядка запуска,
всё последовательно. systemd принёс параллельный запуск, зависимости между службами, единый журнал
и общий язык unit-файлов — и заодно один из самых громких холиваров в истории Linux, потому что
забрал себе слишком много (журналы, сеть, монтирование, cron-подобные таймеры).

Практический след для вас: `systemctl` и `journalctl` вы будете открывать не тогда, когда пишете
службу, а тогда, когда ваш обучающий скрипт на сервере «почему-то не работает». Первые две команды
в такой ситуации — `systemctl status ваш-сервис` и `journalctl -u ваш-сервис -n 50`.

</details>


## Дополнительно

Дальше — справочник: на занятии эти разделы не показывают, но они пригодятся при решении задач и на
своей машине.


### Старые и специализированные инструменты

`venv` входит в Python, `virtualenv` устанавливается отдельным пакетом. Оба создают окружение на основе выбранного интерпретатора; такие команды встречаются в старых проектах и нужны, когда `uv` на машине нет:

```bash
python3.12 -m venv .venv                          # окружение средствами самого Python
python3 -m virtualenv --python=python3.12 .venv   # то же самое сторонним инструментом
source .venv/bin/activate                         # активация — как в разделе 3
python -m pip install -r requirements.txt         # установка зависимостей из списка
python -m pip freeze > requirements.txt           # снимок установленных версий
```

`requirements.txt` — просто список пакетов с версиями, без разделения на прямые и транзитивные зависимости и без хэшей. Это и есть формат, из которого чаще всего приходится восстанавливать чужое окружение. `uv` умеет работать и с ним: `uv venv && uv pip install -r requirements.txt`.


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Про `conda`: в ML-мире она распространена, умеет ставить не только Python-пакеты, но и нативные
библиотеки, и вы обязательно встретите проекты с `environment.yml`. **В нашем курсе conda не
используется** — окружения собираем через `python -m venv` и `uv` (об этом сказано и в плане курса,
`seminars/README.md`). Причина простая: два менеджера окружений в одной голове первокурсника — это
на один слой путаницы больше, а всё, что нужно курсу, `uv` делает быстрее.

</details>


### Версии Python в `uv`

Команды решают разные задачи:

- `uv python install 3.12` — установить управляемый uv интерпретатор;
- `uv python pin 3.12` — записать требование проекта в `.python-version`;
- `uv python find 3.12` — показать выбранный путь;
- `uv python list` — показать найденные и доступные версии.

`pin` не создаёт `.venv` и не устанавливает зависимости.


### Обновление одной зависимости

`uv lock --upgrade-package requests` разрешает uv выбрать новую версию `requests` в пределах ограничений `pyproject.toml`. Остальные зафиксированные пакеты сохраняются, если обновление `requests` не требует их изменения. `uv sync` применяет новый lock-файл к `.venv`.

Перед обновлением полезно сохранить старый `uv.lock`, после — посмотреть `diff` и запустить проверки проекта.


### Подключение APT-репозитория

APT получает пакеты из настроенных источников. Стороннему источнику нужны адрес и ключ проверки подписи. Ниже — последовательность для Docker на Ubuntu; **это справочный рецепт, на занятии мы его не выполняем**: он меняет общесистемную конфигурацию, и делать это стоит осознанно и на своей машине.

Шаг 1 — то, чем скачивают и проверяют: сертификаты и `curl`.

```bash
sudo apt update
sudo apt install ca-certificates curl
```

Шаг 2 — ключ, которым подписан репозиторий. Без него apt откажется брать оттуда пакеты.

```bash
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc
```

Шаг 3 — описание источника. `URIs` — адрес, `Suites` — версия Ubuntu, `Components` — ветка репозитория, `Architectures` — архитектура, `Signed-By` — тот самый ключ.

```bash
sudo tee /etc/apt/sources.list.d/docker.sources <<EOF
Types: deb
URIs: https://download.docker.com/linux/ubuntu
Suites: $(source /etc/os-release && echo "${UBUNTU_CODENAME:-$VERSION_CODENAME}")
Components: stable
Architectures: $(dpkg --print-architecture)
Signed-By: /etc/apt/keyrings/docker.asc
EOF
```

Шаг 4 — перечитать индекс и поставить пакеты.

```bash
sudo apt update
sudo apt install docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin
```

Команды со стороннего сайта перед запуском сверяют с его актуальной официальной инструкцией.


### Что такое unit

Unit-файл — конфигурация systemd. Service-unit задаёт, какую команду запустить и как трактовать её выполнение:

```ini
[Unit]
Description=Show service user

[Service]
Type=oneshot
ExecStart=/usr/bin/id
```

`Type=oneshot` означает, что systemd запускает одно действие, ждёт его завершения и не ожидает постоянно работающий процесс. `ExecStart=/usr/bin/id` задаёт запускаемую команду абсолютным путём. Вывод команды попадает в журнал unit. `systemctl cat NAME.service` показывает основной unit-файл и его drop-in-дополнения — отдельные файлы, которые уточняют или переопределяют настройки. После изменения unit-файлов менеджер перечитывает их через `systemctl daemon-reload`.


### Системные и пользовательские службы

`systemctl --system` обращается к системному менеджеру systemd. Это режим по умолчанию: такие службы работают для всей машины, а изменение и управление ими обычно требуют прав администратора.

`systemctl --user` обращается к отдельному менеджеру текущего пользователя. Пользовательские unit-файлы обычно хранятся в `~/.config/systemd/user/`, не требуют `sudo` и управляют только процессами этого пользователя. Их журналы читают через `journalctl --user -u NAME`. Системная и пользовательская службы с одинаковым именем — разные unit, поэтому режим важно указывать последовательно.

После ручного запуска из терминала проект может не заработать как служба: systemd не запускает интерактивную оболочку, поэтому не читает пользовательский `~/.bashrc` и не выполняет `.venv/bin/activate`. Активация `.venv` лишь меняет `PATH` текущей оболочки и не влияет на отдельный процесс службы — ровно то, что показывала схема в разделе 3. Поэтому в unit-файле явно задают рабочий каталог, окружение и путь к интерпретатору:

```ini
[Service]
WorkingDirectory=/home/student/project
EnvironmentFile=/home/student/project/app.env
ExecStart=/home/student/project/.venv/bin/python /home/student/project/app.py
```

Для пользовательской службы, которая должна работать после выхода пользователя, может потребоваться lingering — разрешение оставлять пользовательский менеджер запущенным без активной сессии. Администратор включает его через `loginctl enable-linger USER`.
